In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np

BASE = '/content/drive/MyDrive/ecommerce-etl-pipeline'
RAW  = f'{BASE}/data/raw'
PROCESSED = f'{BASE}/data/processed'

In [5]:
orders    = pd.read_csv(f'{RAW}/olist_orders_dataset.csv')
customers = pd.read_csv(f'{RAW}/olist_customers_dataset.csv')
order_items = pd.read_csv(f'{RAW}/olist_order_items_dataset.csv')
payments  = pd.read_csv(f'{RAW}/olist_order_payments_dataset.csv')
reviews   = pd.read_csv(f'{RAW}/olist_order_reviews_dataset.csv')
products  = pd.read_csv(f'{RAW}/olist_products_dataset.csv')
sellers   = pd.read_csv(f'{RAW}/olist_sellers_dataset.csv')
geolocation = pd.read_csv(f'{RAW}/olist_geolocation_dataset.csv')
category_translation = pd.read_csv(f'{RAW}/product_category_name_translation.csv')

print("All files loaded!")

All files loaded!


In [7]:
# Fix date columns in orders
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("Orders date types after fix:")
print(orders[date_columns].dtypes)

# Fix date columns in order_items
date_columns = [
    'shipping_limit_date'
]

for col in date_columns:
    order_items[col] = pd.to_datetime(order_items[col])

print("Order_items date types after fix:")
print(order_items[date_columns].dtypes)

Orders date types after fix:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object
Order_items date types after fix:
shipping_limit_date    datetime64[ns]
dtype: object


In [8]:
print("Before filter:", orders.shape)

orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

print("After filter:", orders_delivered.shape)
print("Rows removed:", orders.shape[0] - orders_delivered.shape[0])

Before filter: (99441, 8)
After filter: (96478, 8)
Rows removed: 2963


In [ ]:
# WHY WE USE .copy() HERE
#
# When we filter a dataframe like this:
# orders_delivered = orders[orders['order_status'] == 'delivered']
#
# orders_delivered is NOT a new table — it is just a window
# into the original orders table. If we try to modify it later,
# Python throws a SettingWithCopyWarning because we are
# accidentally modifying the original data too.
#
# By adding .copy() we create a completely independent table.
# Now we can clean and modify orders_delivered freely
# without touching the original orders data.
#
# RULE: Always use .copy() after filtering a dataframe
# that you plan to modify later. This is standard
# professional practice in data engineering.

In [10]:
print("Missing delivery dates in delivered orders:")
print(orders_delivered['order_delivered_customer_date'].isnull().sum())

orders_delivered = orders_delivered.dropna(
    subset=['order_delivered_customer_date']
)
print("Final orders shape:", orders_delivered.shape)

Missing delivery dates in delivered orders:
8
Final orders shape: (96470, 8)


In [11]:
print("Before rename:")
print(products.columns.tolist())

products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

print("\nAfter rename:")
print(products.columns.tolist())

Before rename:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

After rename:
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [12]:
print("Before fill:")
print(reviews[['review_comment_title',
               'review_comment_message']].isnull().sum())

reviews['review_comment_title'] = \
    reviews['review_comment_title'].fillna('No title')

reviews['review_comment_message'] = \
    reviews['review_comment_message'].fillna('No comment')

print("\nAfter fill:")
print(reviews[['review_comment_title',
               'review_comment_message']].isnull().sum())

Before fill:
review_comment_title      87656
review_comment_message    58247
dtype: int64

After fill:
review_comment_title      0
review_comment_message    0
dtype: int64


In [13]:
print("Before:", geolocation.shape)

geolocation_clean = geolocation.drop_duplicates(
    subset=['geolocation_zip_code_prefix']
)

print("After:", geolocation_clean.shape)
print("Duplicates removed:",
      geolocation.shape[0] - geolocation_clean.shape[0])

Before: (1000163, 5)
After: (19015, 5)
Duplicates removed: 981148


In [14]:
products_clean = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

print("Products shape after merge:", products_clean.shape)
print(products_clean[['product_category_name',
                       'product_category_name_english']].head(10))

Products shape after merge: (32951, 10)
   product_category_name product_category_name_english
0             perfumaria                     perfumery
1                  artes                           art
2          esporte_lazer                sports_leisure
3                  bebes                          baby
4  utilidades_domesticas                    housewares
5  instrumentos_musicais           musical_instruments
6             cool_stuff                    cool_stuff
7       moveis_decoracao               furniture_decor
8       eletrodomesticos               home_appliances
9             brinquedos                          toys


In [16]:
import os

# Create the processed folder if it doesn't exist
os.makedirs(PROCESSED, exist_ok=True)
print("Folder created:", PROCESSED)

Folder created: /content/drive/MyDrive/ecommerce-etl-pipeline/data/processed


In [17]:
orders_delivered.to_csv(f'{PROCESSED}/orders_clean.csv', index=False)
customers.to_csv(f'{PROCESSED}/customers_clean.csv', index=False)
order_items.to_csv(f'{PROCESSED}/order_items_clean.csv', index=False)
payments.to_csv(f'{PROCESSED}/payments_clean.csv', index=False)
reviews.to_csv(f'{PROCESSED}/reviews_clean.csv', index=False)
products_clean.to_csv(f'{PROCESSED}/products_clean.csv', index=False)
sellers.to_csv(f'{PROCESSED}/sellers_clean.csv', index=False)
geolocation_clean.to_csv(f'{PROCESSED}/geolocation_clean.csv', index=False)

print("All cleaned files saved to /data/processed/")

All cleaned files saved to /data/processed/
